# Heston–Merton advanced calibration & Monte Carlo — period 1-day 1-minute (2022-10-21)

> **This is not a 2-year regime file.** The evaluation window is **Friday 2022-10-21** (one RTH session; lookback uses prior 1-minute bars on file). Calibration lookback default = **1 hour** (60 trading minutes). Rolling default = **minutely**. Evaluation is **2022-10-21** only. §2 shows **listed** strikes from `short_interval` (same ATM/DTE filters as the 2-year files). §6 is a reminder, matching the 2-year advanced notebooks (LSM lives in the Method A file).

**Period file:** **Friday 2022-10-21 (1-minute bars; 1 RTH session)**.

| Role | Ticker |
|------|--------|
| Primary | **SPY** |
| Secondary | AAPL |
| Secondary | MSFT |

**Calibration method:** **Method B** (light Monte Carlo likelihood over latent variance).  
Basic moments-only Method A lives in `../heston merton notebook/`.

**Section roles**
- **§4 Calibration only:** lookback / rolling, **Reestimate**, inspect estimated parameters (no Monte Carlo plots here).
- **§5 Monte Carlo only:** Start / Restart; simulated paths and history comparison.

True rolling rule: at each update, re-estimate from the current window and use those params for the next MC segment. See `ROLLING_CALIBRATION.md`.


## 0. Setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets
%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA = Path("..") / ".." / "research" / "data"
TICKERS = ["AAPL", "MSFT", "SPY"]
BARS_PER_DAY = 390  # regular session 09:30–16:00 ET
N_DAYS = 252 * BARS_PER_DAY  # annualize 1-minute log returns
N_STEPS = 5000  # keep the full 1-minute grid (no daily-style subsample)
JUMP_THRESH = 3.0
MIN_WINDOW = 60
# Method B compute budget (estimation MC only)
MC_PATHS_CAL = 32
OPT_MAXITER = 25
EST_SEED = 42
COLORS = {"AAPL": "#1f77b4", "MSFT": "#ff7f0e", "SPY": "#2ca02c"}

WINDOW_OPTIONS = {
    "1 hour": 60,   # trading minutes (not calendar hours)
    "1 day": 390,
}
ROLLING_OPTIONS = ["minutely", "hourly"]

WINDOW_ID = "2022-10-21"
INTRADAY_DIR = DATA / "equity" / "short_interval" / "prices_1min_rth"
RATES_PATH = DATA / "rates" / "risk_free_dgs3mo_short_interval.csv"
OPT_DIR = DATA / "options" / "processed" / "short_interval"

def _with_option_days(fn, *args, **kwargs):
    """§6 uses a trading-day clock (N=252), same as the 2-year notebooks."""
    global N_DAYS
    saved = N_DAYS
    N_DAYS = 252
    try:
        return fn(*args, **kwargs)
    finally:
        N_DAYS = saved
_frames = []
for _t in TICKERS:
    _p = pd.read_csv(INTRADAY_DIR / f"{_t}.csv", parse_dates=["Datetime"]).set_index("Datetime").sort_index()
    _frames.append(_p["Close"].rename(_t))
GAP_MIN = pd.Timedelta(minutes=2)

def _session_gap(idx) -> pd.Series:
    return pd.Series(idx, index=idx).diff() > GAP_MIN

def stitch_continuous(close: pd.Series) -> pd.Series:
    """Rebuild a gapless trading-time price: overnight/weekend returns are not applied."""
    s = close.dropna()
    r = np.log(s).diff()
    r = r.mask(_session_gap(s.index), 0.0)
    r.iloc[0] = 0.0
    return pd.Series(float(s.iloc[0]) * np.exp(r.cumsum()), index=s.index, name=s.name)

def trading_x(idx) -> np.ndarray:
    return np.arange(len(idx))

def session_starts(idx) -> np.ndarray:
    g = _session_gap(idx).fillna(False).to_numpy()
    return np.flatnonzero(g)

def n_steps_to_expiry(quote_ts, expiry_ts=None) -> int:
    """Remaining 1-minute RTH bars from quote to expiration (PERIOD_END)."""
    q = pd.Timestamp(quote_ts)
    e = pd.Timestamp(expiry_ts) if expiry_ts is not None else PERIOD_END
    idx = prices.index
    n = int(((idx > q) & (idx <= e)).sum())
    return max(n, 2)
prices_raw = pd.concat(_frames, axis=1).sort_index()
PERIOD_START = pd.Timestamp("2022-10-21 09:30:00")
PERIOD_END = pd.Timestamp("2022-10-21 15:59:00")
prices = pd.concat(
    [stitch_continuous(prices_raw[t]).rename(t) for t in TICKERS],
    axis=1,
).sort_index()
period_prices = prices.loc[PERIOD_START:PERIOD_END, TICKERS].copy()
log_returns_all = np.log(prices[TICKERS]).diff()
# Do not treat the stitched 0-return at session joins as a 1-minute observation
for _t in TICKERS:
    _g = _session_gap(prices[_t].dropna().index)
    log_returns_all.loc[_g.reindex(log_returns_all.index, fill_value=False), _t] = np.nan


rolling = {}
cal_meta = {}

print(f"1-min sample: {prices.index.min()} → {prices.index.max()}")
print(
    f"Evaluation window: {len(period_prices)} 1-minute bars "
    f"({period_prices.index.min()} → {period_prices.index.max()}) — NOT a 2-year regime"
)
print(f"Method B budget: M={MC_PATHS_CAL} latent paths, maxiter={OPT_MAXITER}, seed={EST_SEED}")
period_prices.head()

plt.close("all")
plt.ioff()


## 1. Stock price trends (2022-10-21, continuous RTH)

Overnight/weekend hours are dropped and prices are stitched into a continuous trading-time series. Dotted lines mark session joins. The return/vol panel uses 1-minute log returns on that same series.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    s = period_prices[ticker].dropna()
    x = trading_x(s.index)
    ax.plot(x, s.values, color=COLORS[ticker], lw=1.4)
    for br in session_starts(s.index):
        ax.axvline(br, color="0.75", lw=0.8, ls=":")
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("Close")
    ax.set_title(f"{ticker} ({role}) — continuous trading-time close (RTH only)")
axes[-1].set_xlabel("trading minute (overnight/weekend removed)")
fig.suptitle("Stock price trends — 2022-10-21, continuous RTH", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
display(period_prices.describe().T[["count", "mean", "min", "max"]].round(4))

# Realized volatility from 1-minute log returns (annualized with 252 × 390)
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
vol_rows = []
for ax, ticker in zip(axes, TICKERS):
    r = log_returns_all[ticker].loc[PERIOD_START:PERIOD_END].dropna()
    rv = r.rolling(30, min_periods=10).std() * np.sqrt(N_DAYS)
    ax.plot(trading_x(rv.index), rv.values, color=COLORS[ticker], lw=1.0)
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("σ̂ (ann.)")
    ax.set_title(f"{ticker} ({role}) — 30-min rolling vol from 1-min returns")
    vol_rows.append({
        "ticker": ticker,
        "n_bars": int(r.shape[0]),
        "sigma_1min": float(r.std(ddof=1)),
        "sigma_ann": float(r.std(ddof=1) * np.sqrt(N_DAYS)),
        "mu_ann": float(r.mean() * N_DAYS),
    })
axes[-1].set_xlabel("trading minute")
fig.suptitle("Minute-level realized volatility — continuous RTH", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
display(pd.DataFrame(vol_rows).set_index("ticker").round(6))


## 2. Strike prices in this period

Systematic sample, same rule as every model: **every 5 minutes** in RTH (~78 observations; 09:30–15:55). Nearest-ATM listed call, DTE 7–60, listed expiry. No random dates. Quote times are unique 1-minute stamps. RMSE is percentage RMSE vs market. 2022-10-21 is the monthly third-Friday expiry, so it is a valid 1-day window. Quote times are unique RTH 1-minute stamps.


In [ ]:
import sys
_SCRIPTS = Path("..") / "scripts"
if str(_SCRIPTS.resolve()) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS.resolve()))
from american_lsm import load_calls, sample_listed_minute_calls

_section2 = {}
for ticker in TICKERS:
    df = sample_listed_minute_calls(
        load_calls(DATA, ticker, panel="short_interval"),
        prices[ticker],
        PERIOD_START,
        PERIOD_END,
        
    )
    _section2[ticker] = df
    uniq_t = df["trading_date"].nunique() if len(df) else 0
    n_exp = df["expiration"].dt.normalize().nunique() if len(df) else 0
    display(Markdown(
        f"### {ticker} — {len(df)} contracts, {uniq_t} unique 1-minute times, "
        f"{n_exp} listed expiries"
    ))
    if len(df):
        print("times unique:", bool(df["trading_date"].is_unique),
              "| minute grid:", bool((df["trading_date"].dt.floor("min") == df["trading_date"]).all()),
              "| expiries:", sorted({pd.Timestamp(x).normalize().date() for x in df["expiration"]}))
        cols = [c for c in ["trading_date", "S_t", "K", "expiration", "dte", "n_steps", "r", "moneyness", "option_price"] if c in df.columns]
        display(df[cols].round(4))
    else:
        print("No contracts.")


## 3. Estimation formulas (Heston–Merton — Method B)

Same model as the basic folder:

$$dv_t=\kappa(\theta-v_t)\,dt+\xi\sqrt{v_t}\,dW_t^v,\qquad
\frac{dS_t}{S_{t-}}=(\mu-\lambda\kappa_J)\,dt+\sqrt{v_t}\,dW_t^S+(e^J-1)\,dN_t$$

with \(\mathrm{Corr}(dW^S,dW^v)=\rho\) and \(\kappa_J=e^{\mu_J+\sigma_J^2/2}-1\).

### Method B pipeline (per lookback window)

1. **Warm start (Method A moments)** — \(\mu,\kappa,\theta,\xi,\rho,v_0\) from \(RV_t=r_t^2\); jumps via \(|r_t|>3\hat\sigma_{\mathrm{day}}\).
2. **Hold** \((\mu,\lambda,\mu_J,\sigma_J,\kappa_J)\) from Method A.
3. **Refine** \((\kappa,\theta,\xi,\rho,v_0)\) by light Monte Carlo likelihood:
   - Simulate \(M=32\) latent variance paths on the 1-minute grid (fixed seed).
   - Non-jump days: Gaussian log-density of \(r_t\) given current \(v\); evolve \(v\) with \(Z_v\) correlated to the return residual through \(\rho\).
   - Minimize average negative log-likelihood with a capped coordinate/random search (`maxiter=25`, pure NumPy — no SciPy required).

No research-scale path count in calibration. §5 Monte Carlo is separate.

**Six quantities in §4:** \(\mu,\theta,\kappa,\xi,\rho,\lambda\).


## 4. Calibration only — 2022-10-21

Sliders + **Reestimate**. Shows **only** the rolling parameter graphs (no tables).  
Default rolling = **hourly** (Method B is heavier than moments). Daily works but is slower.  
Monte Carlo vs history is in **§5** only.


In [ ]:
def estimate_heston_merton_moments(log_rets: pd.Series, jump_thresh: float = JUMP_THRESH):
    """Method A warm start: moments + jump threshold (no RNG)."""
    x = log_rets.dropna().astype(float)
    n = int(x.shape[0])
    nan = (np.nan,) * 10 + (n,)
    if n < MIN_WINDOW:
        return nan

    sigma_day = float(x.std(ddof=1))
    if not np.isfinite(sigma_day) or sigma_day <= 0:
        return nan

    jump_mask = np.abs(x.values) > jump_thresh * sigma_day
    jumps = x.iloc[jump_mask]
    normal = x.iloc[~jump_mask]
    base = normal if int(normal.shape[0]) >= 2 else x
    mu = float(base.mean() * N_DAYS)

    rv = x**2
    theta = float(rv.mean() * N_DAYS)
    recent = rv.iloc[-min(21, n):]
    v0 = float(recent.mean() * N_DAYS)
    if not np.isfinite(theta) or theta <= 0:
        return nan
    if not np.isfinite(v0) or v0 <= 0:
        v0 = theta

    rho1 = rv.autocorr(lag=1)
    dt = 1.0 / N_DAYS
    if rho1 is None or not np.isfinite(rho1) or rho1 <= 1e-6:
        kappa = 2.0
    elif rho1 >= 0.999:
        kappa = 0.05
    else:
        kappa = float(-np.log(float(rho1)) / dt)
    kappa = float(np.clip(kappa, 0.05, 20.0))

    v_ann = (rv * N_DAYS).astype(float)
    dv = v_ann.diff().dropna()
    v_lag = v_ann.loc[dv.index]
    drift = kappa * (theta - v_lag.values) * dt
    resid = dv.values - drift
    mean_v = float(np.mean(v_lag.values))
    var_resid = float(np.var(resid, ddof=1)) if resid.size >= 2 else np.nan
    if mean_v > 0 and np.isfinite(var_resid) and var_resid > 0:
        xi = float(np.sqrt(var_resid / (mean_v * dt)))
    else:
        xi = 0.5
    xi = float(np.clip(xi, 0.05, 3.0))

    aligned = pd.concat([x.rename("r"), v_ann.diff().rename("dv")], axis=1).dropna()
    if len(aligned) >= 5:
        rho = float(aligned["r"].corr(aligned["dv"]))
    else:
        rho = -0.5
    if not np.isfinite(rho):
        rho = -0.5
    rho = float(np.clip(rho, -0.99, 0.99))

    years = n / float(N_DAYS)
    n_jumps = int(jump_mask.sum())
    lam = float(n_jumps / years) if years > 0 else 0.0
    if n_jumps >= 2:
        mu_j = float(jumps.mean())
        sigma_j = float(jumps.std(ddof=1))
    elif n_jumps == 1:
        mu_j = float(jumps.iloc[0])
        sigma_j = 0.0
    else:
        mu_j = 0.0
        sigma_j = 0.0
    if not np.isfinite(sigma_j) or sigma_j < 0:
        sigma_j = 0.0
    kappa_j = float(np.exp(mu_j + 0.5 * sigma_j**2) - 1.0)
    return mu, kappa, theta, xi, rho, v0, lam, mu_j, sigma_j, kappa_j, n


def _pack_heston(kappa, theta, xi, rho, v0):
    """Unconstrained vector for optimizer."""
    return np.array([
        np.log(max(kappa, 1e-4)),
        np.log(max(theta, 1e-8)),
        np.log(max(xi, 1e-4)),
        np.arctanh(np.clip(rho, -0.999, 0.999)),
        np.log(max(v0, 1e-8)),
    ], dtype=float)


def _unpack_heston(z):
    kappa = float(np.clip(np.exp(z[0]), 0.05, 20.0))
    theta = float(np.clip(np.exp(z[1]), 1e-6, 1.0))
    xi = float(np.clip(np.exp(z[2]), 0.05, 3.0))
    rho = float(np.clip(np.tanh(z[3]), -0.99, 0.99))
    v0 = float(np.clip(np.exp(z[4]), 1e-6, 1.0))
    return kappa, theta, xi, rho, v0


def _heston_avg_nll(z, r, jump_mask, mu, lam, kappa_j, M, seed):
    """Average negative log-likelihood over M latent variance paths."""
    kappa, theta, xi, rho, v0 = _unpack_heston(z)
    n = r.shape[0]
    dt = 1.0 / N_DAYS
    rng = np.random.default_rng(seed)
    v = np.full(M, max(v0, 1e-12), dtype=float)
    total_ll = np.zeros(M, dtype=float)
    rho = float(np.clip(rho, -0.999, 0.999))
    s_orth = np.sqrt(max(1.0 - rho * rho, 0.0))

    for t in range(n):
        v_pos = np.maximum(v, 1e-12)
        mean = (mu - lam * kappa_j - 0.5 * v_pos) * dt
        vol = np.sqrt(np.maximum(v_pos * dt, 1e-18))
        if not jump_mask[t]:
            z_s = (r[t] - mean) / vol
            total_ll += -0.5 * (np.log(2.0 * np.pi) + 2.0 * np.log(vol) + z_s * z_s)
        else:
            z_s = rng.standard_normal(M)
        z_indep = rng.standard_normal(M)
        z_v = rho * z_s + s_orth * z_indep
        v = v_pos + kappa * (theta - v_pos) * dt + xi * np.sqrt(v_pos) * np.sqrt(dt) * z_v
        v = np.maximum(v, 1e-12)

    avg_ll = float(np.mean(total_ll))
    if not np.isfinite(avg_ll):
        return 1e12
    return -avg_ll


def _refine_heston_mc(z0, r, jump_mask, mu, lam, kappa_j, M, seed, maxiter=OPT_MAXITER):
    """Capped coordinate + random search (no SciPy). Starts at Method A packing z0."""
    z = np.array(z0, dtype=float, copy=True)
    best = float(_heston_avg_nll(z, r, jump_mask, mu, lam, kappa_j, M, seed))
    rng = np.random.default_rng(seed + 17)
    # relative step sizes in unconstrained space
    scales = np.array([0.15, 0.20, 0.15, 0.20, 0.20], dtype=float)
    for it in range(maxiter):
        improved = False
        # coordinate descent
        for j in range(5):
            for sign in (-1.0, 1.0):
                cand = z.copy()
                cand[j] = cand[j] + sign * scales[j]
                val = float(_heston_avg_nll(cand, r, jump_mask, mu, lam, kappa_j, M, seed))
                if val < best:
                    best = val
                    z = cand
                    improved = True
        # one random probe
        cand = z + rng.normal(0.0, 1.0, size=5) * scales * 0.5
        val = float(_heston_avg_nll(cand, r, jump_mask, mu, lam, kappa_j, M, seed))
        if val < best:
            best = val
            z = cand
            improved = True
        if not improved:
            scales *= 0.7
            if np.max(scales) < 1e-3:
                break
    return z, best


def estimate_heston_merton_params(log_rets: pd.Series, jump_thresh: float = JUMP_THRESH, warm=None):
    """Method B: Method A warm start + light MC likelihood for (κ,θ,ξ,ρ,v0)."""
    x = log_rets.dropna().astype(float)
    n = int(x.shape[0])
    nan = (np.nan,) * 10 + (n,)
    if n < MIN_WINDOW:
        return nan

    a = estimate_heston_merton_moments(x, jump_thresh)
    if not np.isfinite(a[0]):
        return nan
    mu, kappa0, theta0, xi0, rho0, v00, lam, mu_j, sigma_j, kappa_j, _ = a

    sigma_day = float(x.std(ddof=1))
    jump_mask = np.abs(x.values) > jump_thresh * max(sigma_day, 1e-12)
    r = x.values.astype(float)

    if warm is not None and np.all(np.isfinite(warm)):
        kappa0, theta0, xi0, rho0, v00 = [float(v) for v in warm]

    z0 = _pack_heston(kappa0, theta0, xi0, rho0, v00)

    try:
        z_hat, nll = _refine_heston_mc(z0, r, jump_mask, mu, lam, kappa_j, MC_PATHS_CAL, EST_SEED, OPT_MAXITER)
        if np.isfinite(nll):
            kappa, theta, xi, rho, v0 = _unpack_heston(z_hat)
        else:
            kappa, theta, xi, rho, v0 = kappa0, theta0, xi0, rho0, v00
    except Exception:
        kappa, theta, xi, rho, v0 = kappa0, theta0, xi0, rho0, v00

    return mu, kappa, theta, xi, rho, v0, lam, mu_j, sigma_j, kappa_j, n


def _slice_window(rets: pd.Series, end: pd.Timestamp, n_bars) -> pd.Series:
    """Last n trading-time bars ending at `end` (calendar gaps already removed)."""
    sub = rets.loc[rets.index <= pd.Timestamp(end)]
    n = int(n_bars)
    return sub.iloc[-n:] if len(sub) else sub


def calibrate_ticker(ticker: str, window_label: str, rolling_mode: str) -> pd.DataFrame:
    rets = log_returns_all[ticker].dropna()
    offset = WINDOW_OPTIONS[window_label]
    rows = []
    warm = None

    period_idx = rets.loc[(rets.index >= PERIOD_START) & (rets.index <= PERIOD_END)].index
    if rolling_mode == "minutely":
        update_dates = period_idx
    elif rolling_mode == "hourly":
        hours = period_idx.floor("h")
        update_dates = pd.DatetimeIndex(
            [period_idx[hours == h].max() for h in hours.unique()]
        ).sort_values()
    else:
        update_dates = pd.DatetimeIndex([period_idx[0]])

    for t_u in update_dates:
        window = _slice_window(rets, pd.Timestamp(t_u), offset)
        mu, kappa, theta, xi, rho, v0, lam, mu_j, sigma_j, kappa_j, n = estimate_heston_merton_params(
            window, warm=warm
        )
        if n < MIN_WINDOW or not np.isfinite(mu):
            continue
        warm = (kappa, theta, xi, rho, v0)
        rows.append({
            "date": pd.Timestamp(t_u),
            "window_start": window.index.min(),
            "window_end": window.index.max(),
            "n_days": n,
            "mu": mu,
            "kappa": kappa,
            "theta": theta,
            "xi": xi,
            "rho": rho,
            "v0": v0,
            "lam": lam,
            "mu_j": mu_j,
            "sigma_j": sigma_j,
            "kappa_j": kappa_j,
        })
    return pd.DataFrame(rows)


def _mc_time_grid(hist: pd.Series, n_steps: int = N_STEPS):
    hist = hist.dropna()
    n_full = len(hist)
    if n_full <= n_steps + 1:
        return hist
    idx = np.linspace(0, n_full - 1, n_steps + 1)
    idx = np.rint(idx).astype(int)
    for i in range(1, len(idx)):
        if idx[i] <= idx[i - 1]:
            idx[i] = min(idx[i - 1] + 1, n_full - 1)
    return hist.iloc[idx]


def param_schedule_for_steps(ticker: str, cal_table: pd.DataFrame):
    hist = _mc_time_grid(period_prices[ticker], N_STEPS)
    dates = hist.index
    n_steps = len(dates) - 1
    cal = cal_table.sort_values("date").reset_index(drop=True)
    cal_dates = pd.to_datetime(cal["date"]).to_numpy()
    cols = ["mu", "kappa", "theta", "xi", "rho", "v0", "lam", "mu_j", "sigma_j", "kappa_j"]
    arrs = {c: cal[c].to_numpy(dtype=float) for c in cols}
    steps = {c: np.empty(n_steps, dtype=float) for c in cols}
    for i in range(n_steps):
        idx = np.searchsorted(cal_dates, np.datetime64(dates[i]), side="right") - 1
        if idx < 0:
            idx = 0
        for c in cols:
            steps[c][i] = arrs[c][idx]
    return (
        dates,
        steps["mu"], steps["kappa"], steps["theta"], steps["xi"], steps["rho"],
        steps["v0"], steps["lam"], steps["mu_j"], steps["sigma_j"], steps["kappa_j"],
        float(hist.iloc[0]), hist,
    )


def _show_fig(fig):
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def plot_rolling_paths(rolling_dict: dict, window_label: str, rolling_mode: str):
    panels = [
        ("mu", "μ̂ (annual)", "Estimated drift"),
        ("theta", "θ̂ (var)", "Long-run variance"),
        ("kappa", "κ̂", "Mean-reversion speed"),
        ("xi", "ξ̂", "Vol-of-vol"),
        ("rho", "ρ̂", "Price–vol correlation"),
        ("lam", "λ̂ (jumps/year)", "Estimated jump intensity"),
    ]
    with plt.ioff():
        fig, axes = plt.subplots(6, 1, figsize=(11, 14), sharex=True)
        for ax, (col, ylab, title) in zip(axes, panels):
            for t in TICKERS:
                r = rolling_dict[t]
                x = np.array([
                period_prices.index.get_indexer([pd.Timestamp(d)], method="pad")[0]
                for d in r["date"]
            ])
                mark = "o" if len(r) < 40 else None
                ax.plot(x, r[col], lw=1.2, label=t, color=COLORS[t], marker=mark, ms=3)
            if col in {"mu", "rho"}:
                ax.axhline(0, color="0.5", lw=0.8)
            ax.set_ylabel(ylab)
            ax.set_title(f"{title} — {rolling_mode}, lookback {window_label} (Method B)")
            ax.legend(frameon=False, ncol=3)
        axes[-1].set_xlabel("Date")
        fig.tight_layout()
    _show_fig(fig)


plt.close("all")
plt.ioff()
rolling = {}
cal_meta = {}

cal_out = widgets.Output(layout=widgets.Layout(width="100%"))
window_slider = widgets.SelectionSlider(
    options=list(WINDOW_OPTIONS.keys()),
    value="1 hour",
    description="Lookback",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
rolling_slider = widgets.SelectionSlider(
    options=ROLLING_OPTIONS,
    value="minutely",
    description="Rolling",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
btn_reestimate = widgets.Button(description="Reestimate", button_style="primary", icon="refresh")

cal_ui = widgets.VBox([
    widgets.HTML(
        "<b>§4 Calibration (Method B)</b> — lookback + rolling, then <b>Reestimate</b>. "
        f"Light MC likelihood: M={MC_PATHS_CAL}, maxiter={OPT_MAXITER}. "
        "Default rolling=hourly. Shows μ̂, θ̂, κ̂, ξ̂, ρ̂, λ̂."
    ),
    window_slider,
    rolling_slider,
    btn_reestimate,
    cal_out,
])


def reestimate(_=None):
    global rolling, cal_meta
    window_label = window_slider.value
    rolling_mode = rolling_slider.value
    rolling = {t: calibrate_ticker(t, window_label, rolling_mode) for t in TICKERS}
    cal_meta = {"window_label": window_label, "rolling_mode": rolling_mode, "method": "B"}

    with cal_out:
        clear_output(wait=True)
        display(Markdown(
            f"**Calibration updated (Method B):** lookback=`{window_label}`, rolling=`{rolling_mode}` "
            f"(n_updates: " + ", ".join(f"{t}={len(rolling[t])}" for t in TICKERS) + ")"
        ))
        plot_rolling_paths(rolling, window_label, rolling_mode)
        display(Markdown(
            "Go to **§5** and click **Start** for one MC pair per company. "
            r"Also estimated: $v_0$, $\mu_J$, $\sigma_J$, $\kappa_J$."
        ))


btn_reestimate.on_click(reestimate)
display(cal_ui)
reestimate()


## 5. Monte Carlo only — one graph pair per company (2022-10-21)

| Left | Right |
|------|--------|
| Monte Carlo paths + median | Median path + 25–75% band vs historical prices |

Uses latest **Reestimate** from §4 (Method B params). **Start** / **Restart** redraw that single pair.

**Stock-path metrics** (printed under each pair)

1. **MAE** — median absolute error of the 50th percentile path vs actual $S_t$ (central-tendency fit).
2. **ICP** — interval coverage probability: share of actual prices that fall inside the 25th–75th percentile band.
3. **Average band width** — mean($p_{75}-p_{25}$); how narrow or wide the model’s uncertainty range is.


In [ ]:
def simulate_heston_merton_rolling(
    mu_step, kappa_step, theta_step, xi_step, rho_step, v0_step,
    lam_step, muj_step, sj_step, kapj_step, S0, n_paths, seed,
):
    rng = np.random.default_rng(seed)
    n_steps = len(mu_step)
    dt = 1.0 / N_DAYS
    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = S0
    v = np.full(n_paths, float(v0_step[0]), dtype=float)

    for i in range(n_steps):
        mu = mu_step[i]
        kappa = kappa_step[i]
        theta = theta_step[i]
        xi = xi_step[i]
        rho = float(np.clip(rho_step[i], -0.999, 0.999))
        lam = max(float(lam_step[i]), 0.0)
        mu_j = muj_step[i]
        sigma_j = max(float(sj_step[i]), 0.0)
        kappa_j = kapj_step[i]

        z_v = rng.standard_normal(n_paths)
        z_indep = rng.standard_normal(n_paths)
        z_s = rho * z_v + np.sqrt(max(1.0 - rho**2, 0.0)) * z_indep

        v_pos = np.maximum(v, 0.0)

        n_jumps = rng.poisson(lam * dt, size=n_paths)
        jump_sizes = np.zeros(n_paths, dtype=float)
        mask = n_jumps > 0
        if mask.any():
            jump_sizes[mask] = (
                n_jumps[mask] * mu_j
                + np.sqrt(n_jumps[mask]) * sigma_j * rng.standard_normal(int(mask.sum()))
            )

        paths[:, i + 1] = paths[:, i] * np.exp(
            (mu - 0.5 * v_pos - lam * kappa_j) * dt
            + np.sqrt(v_pos * dt) * z_s
            + jump_sizes
        )
        v = v + kappa * (theta - v_pos) * dt + xi * np.sqrt(v_pos) * np.sqrt(dt) * z_v
        v = np.maximum(v, 0.0)
    return paths


def _show_fig(fig):
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _draw_ticker_pair(ticker: str, out: widgets.Output, seed: int, n_paths: int = 1000):
    with out:
        clear_output(wait=True)
        if ticker not in rolling or len(rolling[ticker]) == 0:
            display(Markdown("Run **Reestimate** in §4 first."))
            return
        (
            dates_now, mu_now, kappa_now, theta_now, xi_now, rho_now, v0_now,
            lam_now, muj_now, sj_now, kapj_now, S0_now, hist_now,
        ) = param_schedule_for_steps(ticker, rolling[ticker])
        paths = simulate_heston_merton_rolling(
            mu_now, kappa_now, theta_now, xi_now, rho_now, v0_now,
            lam_now, muj_now, sj_now, kapj_now, S0_now, n_paths, seed,
        )
        expected = paths.mean(axis=0)
        p25 = np.percentile(paths, 25, axis=0)
        p50 = np.percentile(paths, 50, axis=0)
        p75 = np.percentile(paths, 75, axis=0)
        _hist = np.asarray(hist_now.values, dtype=float)
        _n = min(len(p50), len(_hist))
        p25, p50, p75, expected, _hist = p25[:_n], p50[:_n], p75[:_n], expected[:_n], _hist[:_n]

        with plt.ioff():
            fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
            tt = trading_x(dates_now)[:_n]
            axes[0].plot(tt, paths[:, :_n].T, color=COLORS[ticker], alpha=0.12, lw=0.7)
            axes[0].plot(tt, p50, color="black", lw=2.0, ls="--", label="median path (50th)")
            axes[0].set_title(f"{ticker}: Heston–Merton Monte Carlo")
            axes[0].set_ylabel("price")
            axes[0].legend(loc="best", frameon=False)

            axes[1].fill_between(tt, p25, p75, color=COLORS[ticker], alpha=0.18, lw=0, zorder=1, label="25–75% range")
            axes[1].plot(tt, _hist, color=COLORS[ticker], lw=1.8, label="historical", zorder=3)
            axes[1].plot(tt, p50, color="black", lw=2.0, ls="--", label="median path (50th)", zorder=4)
            axes[1].set_title(f"{ticker}: median vs history")
            axes[1].set_ylabel("price")
            axes[1].legend(loc="best", frameon=False)
            for ax in axes:
                ax.set_xlabel("trading minute")
            mae = float(np.mean(np.abs(p50 - _hist)))
            icp = float(np.mean((_hist >= p25) & (_hist <= p75)))
            abw = float(np.mean(p75 - p25))
            rmse = float(100.0 * np.sqrt(np.mean(((p50 - _hist) / np.maximum(np.abs(_hist), 1e-8)) ** 2)))
            fig.suptitle(
                f"{ticker} | MAE={mae:.4f} | ICP={100*icp:.1f}% | width={abw:.4f} | seed={seed} | "
                f"{cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')} | Method B",
                fontsize=11,
                y=1.02,
            )
            fig.tight_layout()
        _show_fig(fig)
        display(Markdown(
            f"**MAE (50th vs $S_t$)** = `{mae:.4f}` · **ICP (25–75)** = `{100*icp:.1f}%` · **avg band width** = `{abw:.4f}` · RMSE%(p50) = `{rmse:.2f}%` "
            f"| seed = `{seed}`"
        ))


def make_ticker_panel(ticker: str, n_paths: int = 1000):
    state = {"seed": 42}
    mode = cal_meta.get("rolling_mode", "?")
    win = cal_meta.get("window_label", "?")
    out = widgets.Output(layout=widgets.Layout(width="100%"))
    btn_start = widgets.Button(description="Start", button_style="success", icon="play")
    btn_restart = widgets.Button(description="Restart", button_style="warning", icon="refresh")
    info = widgets.HTML(f"<b>{ticker}</b> — one graph pair | lookback={win}, mode={mode} | Method B")

    busy = {"on": False}

    def on_start(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    def on_restart(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            state["seed"] = int(np.random.default_rng().integers(0, 1_000_000_000))
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    btn_start.on_click(on_start)
    btn_restart.on_click(on_restart)
    return widgets.VBox([info, widgets.HBox([btn_start, btn_restart]), out])


plt.close("all")
plt.ioff()

mc_host = widgets.VBox([])
children = [widgets.HTML("<b>§5 Monte Carlo — click <i>Start</i> once per company (one pair only)</b>")]
for ticker in TICKERS:
    role = "primary" if ticker == "SPY" else "secondary"
    children.append(widgets.HTML(f"<h4 style='margin:8px 0 4px'>{ticker} ({role})</h4>"))
    children.append(make_ticker_panel(ticker))
mc_host.children = tuple(children)
display(mc_host)


## 6. Reminder

1. **§4:** sliders → **Reestimate** → Method B light-MC likelihood → rolling \(\hat\mu,\hat\theta,\hat\kappa,\hat\xi,\hat\rho,\hat\lambda\) charts.  
2. **§5:** **Start** → Heston–Merton Monte Carlo + expected vs history (one pair per ticker).  
3. **Restart** only changes the random seed.
4. Basic Method A notebooks are in `../heston merton notebook/`.
